# Preparación Automatizada de Rásters MapBiomas Chile (30 m) desde Google Earth Engine

**Uso docente / administración del repositorio `magdzraw/byc-2026`** (Facultad de Ciencias Forestales, Universidad de Concepción).

> **Importante:** Este cuaderno auxiliar está pensado para ejecutarse **una sola vez por el docente/ayudante** con una cuenta activa de Google Earth Engine (proyecto `byc-2026`). Los estudiantes **no necesitan** abrir este cuaderno ni usar GEE: el cuaderno principal (`BYC_Dist_AOO_EOO.ipynb`, Módulo 12) descarga y analiza automáticamente los paquetes `.zip` livianos generados aquí.

### ¿Qué automatiza este cuaderno (Opción A)?
1. Se autentica en GEE con tu proyecto `byc-2026` e inspecciona las bandas oficiales de:
   * **MapBiomas Chile Colección 2.0 (Cobertura y Uso del Suelo, 1999–2024):** hitos configurables (`1999, 2005, 2010, 2015, 2020, 2024`).
   * **MapBiomas Fuego Chile Colección 1.0 (2013–2025):** Frecuencia de incendios, Año del último incendio y Cobertura quemada acumulada.
2. Calcula la huella terrestre chilena de cada especie (*Alerce*, *Araucaria*, *Lleuque*, *Queule*, *Ruil*), divide el área en subtiles alineados a $30\text{ m}$ (podando océano y territorio fuera de Chile), los descarga en paralelo con `ee.data.computePixels` y los cose automáticamente con `rasterio.merge` en GeoTIFFs comprimidos (`DEFLATE` + `TILED`).
3. Empaqueta un único archivo `mapbiomas_<especie>_30m.zip` por especie (~2–15 MB c/u) y opcionalmente lo guarda en tu Google Drive listo para publicar en **GitHub Releases** de `magdzraw/byc-2026`.

In [ ]:
# 1. Instalar dependencias y obtener script oficial del repositorio byc-2026
!pip install earthengine-api rasterio geopandas pyogrio requests shapely --quiet

import os, requests

REPO_RAW = "https://raw.githubusercontent.com/magdzraw/byc-2026/main"
os.makedirs("scripts", exist_ok=True)
os.makedirs("data", exist_ok=True)

for rel in ["scripts/descargar_mapbiomas_gee.py", "data/Coordenadas_codigo.csv"]:
    if not os.path.exists(rel):
        r = requests.get(f"{REPO_RAW}/{rel}", timeout=20)
        r.raise_for_status()
        with open(rel, "wb") as fh:
            fh.write(r.content)
        print(f"Descargado desde GitHub: {rel}")
    else:
        print(f"Ya disponible localmente: {rel}")

In [ ]:
#@title 2. Autenticación en Google Earth Engine (proyecto byc-2026) { display-mode: "form" }
PROJECT_ID = "byc-2026"  #@param {type:"string"}

import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print(f"Google Earth Engine listo con proyecto '{PROJECT_ID}'.")

In [ ]:
#@title 3. Configurar y ejecutar descarga automática por tiles + mosaico 30 m { display-mode: "form" }
ESPECIES = "TODAS"  #@param ["TODAS", "Ruil", "Queule", "Lleuque", "Araucaria", "Alerce"]
ANIOS_LULC = "1999 2005 2010 2015 2020 2024"  #@param {type:"string"}
INCLUIR_FUEGO_COL1 = True  #@param {type:"boolean"}
SOLO_INSPECCIONAR = False  #@param {type:"boolean"}
COPIAR_ZIPS_A_GOOGLE_DRIVE = True  #@param {type:"boolean"}
CARPETA_DRIVE = "/content/drive/MyDrive/ByC_2026_Rasters_MapBiomas"  #@param {type:"string"}

import sys, shlex, shutil, glob
from types import SimpleNamespace
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import descargar_mapbiomas_gee as dmg

args = SimpleNamespace(
    project=PROJECT_ID,
    especies=shlex.split(ESPECIES),
    anios=[int(a) for a in shlex.split(ANIOS_LULC)],
    escala=30.0,
    tile_px=1800,
    margen_deg=0.12,
    workers=4,
    csv_localidades="data/Coordenadas_codigo.csv",
    salida="rasters_mapbiomas",
    asset_lulc="",
    auth_mode="colab",
    sin_fuego=(not INCLUIR_FUEGO_COL1),
    mantener_tiles=False,
    solo_inspeccionar=SOLO_INSPECCIONAR,
)
dmg.ejecutar_descarga(args)

zips = sorted(glob.glob("rasters_mapbiomas/mapbiomas_*_30m.zip"))
print("\nPaquetes generados:")
for z in zips:
    print(f"  - {z} ({os.path.getsize(z)/1e6:.2f} MB)")

if COPIAR_ZIPS_A_GOOGLE_DRIVE and zips and not SOLO_INSPECCIONAR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CARPETA_DRIVE, exist_ok=True)
    for z in zips:
        dst = os.path.join(CARPETA_DRIVE, os.path.basename(z))
        shutil.copy2(z, dst)
        print(f"  Copiado a Drive: {dst}")

### 4. Publicación de los paquetes `.zip` para los estudiantes (1 sola vez)

1. Ve a **Releases → Draft a new release** en [`magdzraw/byc-2026`](https://github.com/magdzraw/byc-2026/releases/new).
2. Crea una etiqueta (por ejemplo `v1.1-mapbiomas-30m`) y adjunta los 5 archivos `mapbiomas_<especie>_30m.zip` generados arriba.
3. Publica el *Release*. A partir de ese momento, **el Módulo 12 de `BYC_Dist_AOO_EOO.ipynb` descargará automáticamente el paquete 30 m + Fuego Col. 1** correspondiente a la especie elegida por cada estudiante, sin que nadie necesite cuenta de Google Earth Engine.